# DeepSeek-V4-Mini Lecture 2: Attention type

1. Standard Attention
2. Random-Mask Attention
3. Free-Mask Attention
4. Top-K key/value Attention
5. Compress Attention
6. Sliding Window Attention

## 0. Config & Data

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(42)

In [2]:
# config 
@dataclass
class AttentionModelArgs:
    vocab_size: int = 100
    dim: int = 512
    head_dim: int = 64
    n_heads: int = 8
    n_kv_heads: int = 8
    window_size: int = 128
    compress_ratios: int = 4 # 0, 4, 12
    attn_top_k: int = 11
    
args = AttentionModelArgs()

# data
bsz = 2
seq_len = 256
X = torch.randn(bsz, seq_len, args.dim)

## 1. Standard Attention

In [3]:
def ScaledDotProductionAttention(Q, K, V, M):
    """
    argps
        input: [B, H, L, D] or [B, L, D]
        output: [B, H, L, D] or [B, L, D]
    """
    S = Q @ K.transpose(-2, -1)
    if M != None:
        S = S.masked_fill(M == False, float('-inf'))
    P = F.softmax(S, dim = -1)
    Z = P @ V
    return Z

M = torch.ones(bsz, seq_len, seq_len)
Z = ScaledDotProductionAttention(X, X, X, M)
print(Z.shape)

torch.Size([2, 256, 512])


In [4]:
class AttentionSingleHead(nn.Module):
    """
        single head attention
    """
    def __init__(self, args):
        super().__init__()
        dim, head_dim, n_heads, n_kv_heads = args.dim, args.head_dim, args.n_heads, args.n_kv_heads
        self.Wq = nn.Linear(dim, n_heads * head_dim, bias=False)
        self.Wk = nn.Linear(dim, n_kv_heads * head_dim, bias=False)
        self.Wv = nn.Linear(dim, n_kv_heads * head_dim, bias=False)
        self.Wo = nn.Linear(dim, dim, bias=False)

    def forward(self, X, mask):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        Z = ScaledDotProductionAttention(Q, K, V, mask)
        O = self.Wo(Z)
        return O
        
model_attn_basic = AttentionSingleHead(args)
O = model_attn_basic(X, M)
print(O.shape)

torch.Size([2, 256, 512])


## 2. Random-Mask Attention

In [5]:
shape = [bsz, seq_len, seq_len]
M_random = torch.randint(0, 2, size=shape, dtype=torch.bool)
O_random_mask = model_attn_basic(X, M_random)
print((M_random.int())[0, :2, :10]) 

tensor([[0, 1, 0, 0, 0, 0, 0, 1, 1, 0],
        [1, 0, 0, 0, 1, 0, 1, 1, 0, 1]], dtype=torch.int32)


## 3. Free-Mask Attention

compute-efficient

In [6]:
def SingleQuerySDPA(q, K, V):
    """
    argps
        input: Q[1, D], K,V[L, D], L is dynamic
        output: [1, D]
    """
    S = q @ K.t()
    P = F.softmax(S, dim = -1)
    Z = P @ V
    return Z

class AttentionFreeMask(AttentionSingleHead):
    """
        single free mask SDPA
    """
    def forward_free_mask(self, X, mask):
        B, L, D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)

        Z = torch.zeros_like(Q)
        for i in range(B):
            for j in range(L):
                idx = torch.where(mask[i, j] != 0)[0].tolist()
                q, K_, V_ = Q[i,[j], :], K[i, idx, :], V[i, idx, :]
                Z[i,j,:] = SingleQuerySDPA(q, K_, V_)
                # break
                
        O = self.Wo(Z)
        return O
    
        
model_attn_free = AttentionFreeMask(args)

O_mask = model_attn_free.forward(X, M_random)
O_free_mask = model_attn_free.forward_free_mask(X, M_random)

print(torch.norm(O_mask - O_free_mask))

tensor(0.0003, grad_fn=<LinalgVectorNormBackward0>)


## 4. Top-K key/value Attention

In [7]:
class AttentionTopK(AttentionSingleHead):
    """
        Attention top_k 
    """
    def forward(self, X, topk = 10, is_post=True):
        B, L, D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)

        Z = torch.zeros_like(Q)
        for i in range(B):
            for j in range(L):
                if is_post: # with attention score
                    S = Q[i,[j], :] @ K[i].t()
                else: # without attention score
                    S = torch.randn(1, L)
                _, idx = torch.topk(S, topk)
                idx = idx[0].tolist()
                q, K_, V_ = Q[i,[j], :], K[i, idx, :], V[i, idx, :]
                Z[i,j,:] = SingleQuerySDPA(q, K_, V_)
                # break
                
        O = self.Wo(Z)
        return O
    
        
model_attn_topk = AttentionTopK(args)

O_topk_post = model_attn_topk(X, args.attn_top_k, is_post=True)
O_topk_pre = model_attn_topk(X, args.attn_top_k, is_post=False)

print(O_topk_post.shape)
print(O_topk_pre.shape)

torch.Size([2, 256, 512])
torch.Size([2, 256, 512])


## 5. Compress Attention

In [8]:
class AttentionCompress(AttentionSingleHead):
    """
        Attention Compress
    """
    def forward(self, X, compress_ratio):
        B, L, D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)

        # Compress
        c, n_c = compress_ratio, L//compress_ratio
        if compress_ratio != 0:
            # mean pooling style compress
            K_ = K.reshape(B, n_c, c, D).mean(dim=2) 
            V_ = V.reshape(B, n_c, c, D).mean(dim=2)

        print('orignal K_len', K.shape[1])
        print('compressd K_len', K_.shape[1])
        print(f'len_q:{L}, len_k:{L}, len_kc:{K_.shape[1]}')

        Z = ScaledDotProductionAttention(Q, K_, V_, None)    
        O = self.Wo(Z)
        return O
    
        
model_attn_compress = AttentionCompress(args)

O_compress = model_attn_compress.forward(X, compress_ratio = 1) # orignal attention
O_compress = model_attn_compress.forward(X, compress_ratio = 4)
O_compress = model_attn_compress.forward(X, compress_ratio = 16)

print(O_compress.shape)

orignal K_len 256
compressd K_len 256
len_q:256, len_k:256, len_kc:256
orignal K_len 256
compressd K_len 64
len_q:256, len_k:256, len_kc:64
orignal K_len 256
compressd K_len 16
len_q:256, len_k:256, len_kc:16
torch.Size([2, 256, 512])


## 6. Sliding Window Attention

In [9]:
class AttentionSlidingWindow(AttentionSingleHead):
    """
        Attention Compress
    """
    def forward(self, X, window_size):
        B, L, D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        
        Z = torch.zeros_like(Q)
        for i in range(B):
            for j in range(L):
                left = max(j, j-window_size)
                right = min(j, j+window_size)
                
                q, K_, V_ = Q[i,[j], :], K[i, left:right, :], V[i, left:right, :]
                Z[i,j,:] = SingleQuerySDPA(q, K_, V_)
  
        O = self.Wo(Z)
        return O
    
        
model_attn_window = AttentionSlidingWindow(args)

O_window = model_attn_window.forward(X, args.window_size) 

print(O_window.shape)

torch.Size([2, 256, 512])


## Thinking

1. When we use mixed attention, we concatenate the basic key and the compressed key.
2. Attention is controlled by selecting key IDs. The general approach is to save all keys for the forward pass, and the keys are managed via a paging scheme(e.g. vLLM)
3. Kernel design: the block attention kernel should skip local attention computation for blocks that have no selected keys.